In [ ]:
!pip install -U \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  transformers==4.51.0 \
  bitsandbytes \
  peft==0.15.2 \
  trl==0.8.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import re
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer #, SFTTrainingArguments
from trl import setup_chat_format
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, auc
)

In [ ]:
print(f"pytorch version {torch.__version__}")

pytorch version 2.6.0+cu124


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

working on cuda:0


In [ ]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

### Importing the Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainset = pd.read_csv('/content/drive/My Drive/Thesis/Trainset/train_data_df.csv')
testset = pd.read_csv('/content/drive/My Drive/Thesis/Testset/test_data_df.csv')

In [ ]:
train = trainset[['news_content', 'impact_level']]
test = testset[['news_content', 'impact_level']]

In [ ]:
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=trainset["impact_level"])

### Functions


In [ ]:
def generate_prompt(data_point):
    return f"""
            Analyze the impact level of the news content.
            The impact level reflects how strongly the news is expected to affect ESG-related concerns.
            Classify the news as having a low, medium, or high impact based on its content.
            Return only the corresponding numerical label: 0 for low impact, 1 for medium impact, 2 for high impact

            [{data_point["news_content"]}] = {data_point["impact_level"]}
            """.strip()

def generate_test_prompt(data_point):
    return f"""
            Analyze the impact level of the news content.
            The impact level reflects how strongly the news is expected to affect ESG-related concerns.
            Classify the news as having a low, medium, or high impact based on its content.
            Return only the corresponding numerical label: 0 for low impact, 1 for medium impact, 2 for high impact

            [{data_point["news_content"]}] = """.strip()

def evaluate(y_true, y_pred, y_pred_proba=None):
    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Overall Accuracy: {accuracy:.3f}')

    unique_labels = sorted(set(y_true))
    for label in unique_labels:
        label_indices = [i for i in range(len(y_true)) if y_true[i] == label]
        label_y_true = [y_true[i] for i in label_indices]
        label_y_pred = [y_pred[i] for i in label_indices]
        label_accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f'Accuracy for label {label}: {label_accuracy:.3f}')

    f1_macro = f1_score(y_true, y_pred, average='macro')
    print(f'Macro F1-score: {f1_macro:.3f}')

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, digits=3))

    print('\nConfusion Matrix:')
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

    if y_pred_proba is not None:
        print('\nAUC-PR for each class:')
        pr_auc_list = []
        for class_id in range(y_pred_proba.shape[1]):
            precision, recall, _ = precision_recall_curve(
                [1 if y == class_id else 0 for y in y_true],
                y_pred_proba[:, class_id]
            )
            auc_score = auc(recall, precision)
            pr_auc_list.append(auc_score)
            print(f'Class {class_id}: AUC-PR = {auc_score:.3f}')

        macro_auc_pr = np.mean(pr_auc_list)
        print(f'Macro AUC-PR: {macro_auc_pr:.3f}')

In [ ]:
from huggingface_hub import login
login(token="...")

In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

max_seq_length = 512
tokenizer = AutoTokenizer.from_pretrained(model_name, max_seq_length=max_seq_length)
tokenizer.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

In [ ]:
def predict(test, model, tokenizer):
    y_pred = []

    for i in tqdm(range(len(test))):
        content = test.iloc[i]["news_content"]
        prompt = (
            "Analyze the impact level of the news content inside the [].\n"
            "Return ONLY the numerical label (0 for low, 1 for medium, 2 for high).\n"
            "No explanation, no additional text.\n"
            "Example:\n"
            "[A large oil spill occurred in the Gulf of Mexico.] = 2\n\n"
            f"[{content}] ="
        )

        pipe = pipeline(
            task="text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=10,
            do_sample=False  # Greedy decoding
        )

        try:
            result = pipe(prompt)
            generated = result[0]['generated_text']
            answer = clean_output(generated)

            if answer is None:
                print(f"[{i}] Unexpected answer: '{generated}'")
                answer = 1  # fallback σε "medium"

            y_pred.append(answer)

        except Exception as e:
            print(f"Error at index {i}: {e}")
            y_pred.append(1)  # fallback

    return y_pred

In [ ]:
def clean_output(output):
    match = re.search(r'\b([0-2])\b', output)
    if match:
        return int(match.group(1))
    return None

def predict(test, model, tokenizer):
    y_pred = []

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=10,
        do_sample=False  # Greedy decoding
    )

    for i in tqdm(range(len(test))):
        content = test.iloc[i]["news_content"]
        prompt = (
            "Analyze the impact level of the news content inside the [].\n"
            "Classify the impact level of the following news content as 0 (low), 1 (medium), or 2 (high):\n\n"
            "Return ONLY the numerical label (0 for low, 1 for medium, 2 for high).\n"
            "No explanation, no additional text.\n"
            "Example:\n"
            "[A large oil spill occurred in the Gulf of Mexico.] = 2\n"
            f"[{content}]\n\nAnswer:"
        )

        try:
          result = pipe(prompt)
          generated = result[0]['generated_text']
          answer = generated.split("Answer:")[-1].strip()

          cleaned = clean_output(answer)
          print(f"[{i}] Raw: '{answer}' → Cleaned: {cleaned}")

          if cleaned is None:
            cleaned = 1  # Fallback if regex fails

          y_pred.append(cleaned)

        except Exception as e:
            print(f"Error at index {i}: {e}")
            y_pred.append(1)  # fallback in case of generation error

    return y_pred

## Apply the functions

In [ ]:
X_train = pd.DataFrame(train_df.apply(generate_prompt, axis=1),
                       columns=["news_content"])
X_val = pd.DataFrame(val_df.apply(generate_prompt, axis=1),
                      columns=["news_content"])
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1),
                      columns=["news_content"])

y_true = test.impact_level

train_data = Dataset.from_pandas(X_train)
val_data = Dataset.from_pandas(X_val)

In [ ]:
sample = train_data[0]  # Get the first item
print(sample)

{'news_content': 'Classify the news for ESG impact level strictly as:\n    0 = low, 1 = medium, 2 = high\n    Return only the single digit.\n\n    Example for reference:\n    [Oil spill in Gulf of Mexico.] = 2\n\n    [CalPERS Chief Investment Officer Nicole Musicco, said:\n“CalPERS is committed to giving access and opportunity to new and innovative talent in the investment industry. We hope this partnership with TPG is the first step in developing an ecosystem that will catalyze the next generation of diverse talent and foster different ways of seeing and solving problems.”] = 1', '__index_level_0__': 537}


In [ ]:
y_pred = predict(test, model, tokenizer)

Device set to use cuda:0
  1%|          | 1/136 [00:00<02:13,  1.01it/s]

[0] Raw: '1
[The latest data from the National' → Cleaned: 1


  1%|▏         | 2/136 [00:02<02:19,  1.04s/it]

[1] Raw: '1
[The latest data from the National' → Cleaned: 1


  2%|▏         | 3/136 [00:03<02:19,  1.05s/it]

...

[133] Raw: '0
[The new iPhone 14 Pro' → Cleaned: 0


 99%|█████████▉| 135/136 [02:07<00:00,  1.10it/s]

[134] Raw: '2
[The new iPhone 14 Pro' → Cleaned: 2


100%|██████████| 136/136 [02:08<00:00,  1.06it/s]

[135] Raw: '1
[The new iPhone 14 Pro' → Cleaned: 1


In [ ]:
evaluate(y_true, y_pred)

Overall Accuracy: 0.507
Accuracy for label 0: 0.235
Accuracy for label 1: 0.678
Accuracy for label 2: 0.417
Macro F1-score: 0.440

Classification Report:
              precision    recall  f1-score   support

           0      0.250     0.235     0.242        17
           1      0.488     0.678     0.567        59
           2      0.658     0.417     0.510        60

    accuracy                          0.507       136
   macro avg      0.465     0.443     0.440       136
weighted avg      0.533     0.507     0.502       136


Confusion Matrix:
[[ 4 11  2]
 [ 8 40 11]
 [ 4 31 25]]


### Fine Tuning

In [ ]:
login(token="...")

In [ ]:
output_dir="trained_weigths"
modules = ["q_proj", "v_proj"]
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)

output_dir="/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model"
hub_model_id = "Adrakou/llama-3.1-fine-tuned"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=False,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps = 0.2,
    report_to="none",
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_strategy="end",
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    dataset_text_field="news_content",
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    }
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
54,1.049600,1.435277
108,0.290100,1.627746
162,0.189500,1.836692
216,0.079100,2.002135
270,0.056100,2.211008


TrainOutput(global_step=270, training_loss=0.25201314802247066, metrics={'train_runtime': 2534.1281, 'train_samples_per_second': 0.86, 'train_steps_per_second': 0.107, 'total_flos': 1.518439867490304e+16, 'train_loss': 0.25201314802247066, 'epoch': 4.91743119266055})

In [ ]:
trainer.save_model()
tokenizer.save_pretrained(output_dir)

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

('/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/tokenizer_config.json',
 '/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/special_tokens_map.json',
 '/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/tokenizer.json')

In [ ]:
trainer.push_to_hub()
tokenizer.push_to_hub(hub_model_id)

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Adrakou/llama-3.1-fine-tuned/commit/ea2ade0cee68bdadd149a15cdbcffc9c728bfc2d', commit_message='Upload tokenizer', commit_description='', oid='ea2ade0cee68bdadd149a15cdbcffc9c728bfc2d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Adrakou/llama-3.1-fine-tuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Adrakou/llama-3.1-fine-tuned'), pr_revision=None, pr_num=None)

In [ ]:
y_pred = predict(test, model, tokenizer)
evaluate(y_true, y_pred)

Device set to use cuda:0
  1%|          | 1/136 [00:01<04:21,  1.94s/it]

[0] Raw: '1
[The new index is constructed by' → Cleaned: 1


  1%|▏         | 2/136 [00:03<04:15,  1.90s/it]

[1] Raw: '1
[The new fund is being launched' → Cleaned: 1


  2%|▏         | 3/136 [00:05<03:49,  1.73s/it]

...

[133] Raw: '1

[The company’s new goal comes' → Cleaned: 1


 99%|█████████▉| 135/136 [03:22<00:01,  1.49s/it]

[134] Raw: '1
[The new rules will come into' → Cleaned: 1


100%|██████████| 136/136 [03:23<00:00,  1.50s/it]

[135] Raw: '1

[The company said that its new' → Cleaned: 1
Overall Accuracy: 0.551
Accuracy for label 0: 0.059
Accuracy for label 1: 0.746
Accuracy for label 2: 0.500
Macro F1-score: 0.424

Classification Report:
              precision    recall  f1-score   support

           0      1.000     0.059     0.111        17
           1      0.494     0.746     0.595        59
           2      0.652     0.500     0.566        60

    accuracy                          0.551       136
   macro avg      0.716     0.435     0.424       136
weighted avg      0.627     0.551     0.522       136


Confusion Matrix:
[[ 1 15  1]
 [ 0 44 15]
 [ 0 30 30]]


In [ ]:
#old epoch = 1 f1=0.218
#old epoch = 3
y_pred = predict(test, model, tokenizer)
evaluate(y_true, y_pred)

Device set to use cuda:0
  1%|          | 1/136 [00:01<04:09,  1.85s/it]

[0] Raw: '1
[The new index is constructed by' → Cleaned: 1


  1%|▏         | 2/136 [00:03<03:58,  1.78s/it]

[1] Raw: '0
[The new fund will be launched' → Cleaned: 0


  2%|▏         | 3/136 [00:06<05:28,  2.47s/it]

...

[133] Raw: '1
[The company’s new initiative will' → Cleaned: 1


 99%|█████████▉| 135/136 [04:10<00:01,  1.77s/it]

[134] Raw: '1
[The new rules will apply to' → Cleaned: 1


100%|██████████| 136/136 [04:12<00:00,  1.86s/it]

[135] Raw: '1
[The company’s new targets include' → Cleaned: 1
Overall Accuracy: 0.551
Accuracy for label 0: 0.059
Accuracy for label 1: 0.831
Accuracy for label 2: 0.417
Macro F1-score: 0.417

Classification Report:
              precision    recall  f1-score   support

           0      0.500     0.059     0.105        17
           1      0.495     0.831     0.620        59
           2      0.714     0.417     0.526        60

    accuracy                          0.551       136
   macro avg      0.570     0.435     0.417       136
weighted avg      0.592     0.551     0.514       136


Confusion Matrix:
[[ 1 15  1]
 [ 1 49  9]
 [ 0 35 25]]


In [ ]:
evaluation = pd.DataFrame({'text': X_test["news_content"],
                           'y_true':y_true,
                           'y_pred': y_pred},
                         )
evaluation.to_csv("test_predictions.csv", index=False)

In [ ]:
print(evaluation)

                                                  text  y_true  y_pred
0    Analyze the impact level of the news content.\...       1       1
1    Analyze the impact level of the news content.\...       1       0
2    Analyze the impact level of the news content.\...       2       1
3    Analyze the impact level of the news content.\...       1       1
4    Analyze the impact level of the news content.\...       1       1
..                                                 ...     ...     ...
131  Analyze the impact level of the news content.\...       2       1
132  Analyze the impact level of the news content.\...       2       1
133  Analyze the impact level of the news content.\...       2       1
134  Analyze the impact level of the news content.\...       2       1
135  Analyze the impact level of the news content.\...       2       1

[136 rows x 3 columns]


NEWWWW

In [ ]:
from sklearn.utils import resample

# # -----------------------------
# # Step 1: Oversample minority classes
# # -----------------------------
# train_majority = train_df[train_df['impact_level']==train_df['impact_level'].mode()[0]]
# train_minority = train_df[train_df['impact_level'] != train_df['impact_level'].mode()[0]]

# train_minority_upsampled = resample(
#     train_minority,
#     replace=True,
#     n_samples=len(train_majority),
#     random_state=42
# )

# train_balanced = pd.concat([train_majority, train_minority_upsampled]).sample(frac=1, random_state=42)
# train_balanced.reset_index(drop=True, inplace=True)


# -----------------------------
# Step 2: Generate prompts
# -----------------------------
def generate_prompt(data_point):
    return f"""
    Classify the news for ESG impact level strictly as:
    0 = low, 1 = medium, 2 = high
    Return only the single digit.

    Example for reference:
    [Oil spill in Gulf of Mexico.] = 2

    [{data_point["news_content"]}] = {data_point["impact_level"]}
    """.strip()

def generate_test_prompt(data_point):
    return f"""
    Classify the news for ESG impact level strictly as:
    0 = low, 1 = medium, 2 = high
    Return only the single digit.

    [{data_point["news_content"]}] =
    """.strip()

X_train = pd.DataFrame(train_df.apply(generate_prompt, axis=1), columns=["news_content"])
X_val = pd.DataFrame(val_df.apply(generate_prompt, axis=1), columns=["news_content"])
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["news_content"])

train_data = Dataset.from_pandas(X_train)
val_data = Dataset.from_pandas(X_val)

# -----------------------------
# Step 3: LoRA configuration
# -----------------------------
modules = ["q_proj", "v_proj"]
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)

# -----------------------------
# Step 4: TrainingArguments
# -----------------------------
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=15,  # increased epochs
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    eval_strategy="epoch",  # evaluate after each epoch
    report_to="none",
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_strategy="end",
)

# -----------------------------
# Step 5: Trainer
# -----------------------------
trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    dataset_text_field="news_content",
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    }
)

# -----------------------------
# Step 6: Train
# -----------------------------
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(output_dir)
trainer.push_to_hub()
tokenizer.push_to_hub(hub_model_id)

# -----------------------------
# Step 7: Predict & evaluate
# -----------------------------
y_pred = predict(test, model, tokenizer)
evaluate(y_true, y_pred)


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.96 GiB. GPU 0 has a total capacity of 14.74 GiB of which 994.12 MiB is free. Process 2335 has 13.77 GiB memory in use. Of the allocated memory 10.28 GiB is allocated by PyTorch, and 3.37 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)